# Location Marginal Pricing - GNN

In [ ]:
import numpy as np
import pandas as pd

from lmp_model.config import PROCESSED_DATA_DIR
from lmp_model.dataset import FEATURES, load_nodes, load_window, query
from lmp_model.splits import walk_forward_folds

folds = walk_forward_folds()
nodes = load_nodes()
pnode_ids = nodes.index
folds[0], len(pnode_ids)

In [ ]:
start = folds[0].train_start + pd.DateOffset(months=6)
w = load_window(start, start + pd.Timedelta(days=7), pnode_ids)
w.x.shape, w.y.shape

In [ ]:
def fit_scaler(x):
    flat = x.reshape(-1, x.shape[-1])
    lo, hi = np.nanpercentile(flat, [0.1, 99.9], axis=0)
    clipped = np.clip(flat, lo, hi)
    return lo, hi, np.nanmean(clipped, axis=0), np.nanstd(clipped, axis=0)


def scale(x, scaler):
    lo, hi, mean, std = scaler
    return (np.clip(x, lo, hi) - mean) / std


scaler = fit_scaler(w.x)
x_scaled = scale(w.x, scaler)
pd.DataFrame({"mean": np.nanmean(x_scaled, axis=(0, 1)), "std": np.nanstd(x_scaled, axis=(0, 1)),
              "min": np.nanmin(x_scaled, axis=(0, 1)), "max": np.nanmax(x_scaled, axis=(0, 1))}, index=FEATURES)

In [ ]:
loss_mask = w.exists & ~np.isnan(w.y)
x_scaled = np.nan_to_num(x_scaled)
loss_mask.mean(), np.isnan(w.area_forecast).mean(), np.isnan(w.weather).mean()

In [ ]:
area_scaler = fit_scaler(w.area_forecast)
area_scaled = np.nan_to_num(scale(w.area_forecast, area_scaler))
weather_scaler = fit_scaler(w.weather)
weather_scaled = np.nan_to_num(scale(w.weather, weather_scaler))
area_scaled.shape, weather_scaled.shape, area_scaled.std(axis=0).round(2), weather_scaled.std(axis=(0, 1)).round(2)

In [ ]:
last_verified = pd.Timestamp(query("SELECT max(time) AS t FROM analytics_lmp.lmp_spread")["t"][0])
full = load_window(folds[0].train_start, last_verified + pd.Timedelta(hours=1), pnode_ids)
np.savez_compressed(
    PROCESSED_DATA_DIR / "train_npz" / "window.npz",
    times=full.times.asi8, x=full.x, y=full.y, exists=full.exists, system=full.system,
    area_forecast=full.area_forecast, node_area=full.node_area, weather=full.weather,
    node_weather_zone=full.node_weather_zone, areas=full.areas.to_numpy(),
    weather_zones=full.weather_zones.to_numpy(),
)